# Aula 5 - Mastering Machine Learning Advanced

## MLOps Básico: do Experimento à Produção

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. IBM Telco Customer Churn — https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113

## Índice

1. [Introdução ao MLOps](#1)
2. [Preparação do Dataset](#2)
3. [Rastreamento de Experimentos com MLflow](#3)
   - 3.1 [Logando parâmetros, métricas e artefatos](#31)
   - 3.2 [Comparando experimentos](#32)
4. [Serialização e Carregamento de Modelos](#4)
5. [Servindo o Modelo com FastAPI](#5)
6. [Monitoramento de Data Drift com Evidently](#6)
7. [Pipeline MLOps Completo](#7)
8. [Conclusão](#8)

# 1. Introdução ao MLOps <a id="1"></a>

Construir um modelo é apenas uma parte do trabalho. A maioria dos modelos de ML nunca chega à produção — e dos que chegam, muitos degradam silenciosamente sem ninguém perceber.

**MLOps** (Machine Learning Operations) é o conjunto de práticas que une desenvolvimento de ML com operações de software, garantindo que modelos sejam implantados de forma confiável, reproducível e monitorada.

![MLOps Lifecycle](https://i.imgur.com/JqFzNUr.png)

**Por que isso importa em Telecom?**

Um modelo de churn treinado em janeiro pode estar desatualizado em julho — os padrões de comportamento do cliente mudam com campanhas, sazonalidade e ações da concorrência. Sem MLOps, você nunca sabe quando o modelo parou de funcionar.

Nesta aula vamos cobrir quatro pilares práticos:

| Pilar | Ferramenta | O que resolve |
|---|---|---|
| **Rastreamento de experimentos** | MLflow | Reprodutibilidade, comparação de modelos |
| **Serialização** | joblib + MLflow | Salvar e carregar modelos de forma confiável |
| **Serving** | FastAPI | Expor previsões como API REST |
| **Monitoramento** | Evidently | Detectar degradação e data drift |

In [ ]:
!pip install mlflow evidently fastapi uvicorn nest-asyncio -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

import mlflow
import mlflow.sklearn

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    roc_auc_score, classification_report
)

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)

# 2. Preparação do Dataset <a id="2"></a>

In [ ]:
# carregando e preparando o dataset Telecom Churn
url = 'https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv'
df = pd.read_csv(url)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['customerID'])

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

X = df[cols_num + cols_cat]
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# preprocessador padrão do curso
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), cols_num),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cols_cat)
])

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')
print(f'Taxa de Churn no treino: {y_train.mean()*100:.1f}%')

# 3. Rastreamento de Experimentos com MLflow <a id="3"></a>

Sem rastreamento, perguntas simples como "qual foi a melhor combinação de hiperparâmetros que testamos na semana passada?" são impossíveis de responder. O **MLflow** registra automaticamente parâmetros, métricas, código e artefatos de cada experimento.

## 3.1 Logando parâmetros, métricas e artefatos <a id="31"></a>

In [ ]:
# configurando o MLflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('telecom-churn-prediction')

print('Experimento configurado: telecom-churn-prediction')

In [ ]:
def treinar_e_logar(nome_run, modelo_clf, params):
    """Treina um modelo e loga tudo no MLflow."""
    with mlflow.start_run(run_name=nome_run):
        # construindo o pipeline completo
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model',        modelo_clf)
        ])

        # treinando
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # calculando métricas
        acc    = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1     = f1_score(y_test, y_pred)
        auc    = roc_auc_score(y_test, y_prob)

        # logando no MLflow
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': acc,
            'recall':   recall,
            'f1_score': f1,
            'roc_auc':  auc
        })
        mlflow.set_tag('dataset',  'TelcoChurn')
        mlflow.set_tag('model',    nome_run)

        # salvando o modelo como artefato
        mlflow.sklearn.log_model(pipeline, artifact_path='model')

        run_id = mlflow.active_run().info.run_id
        print(f'[{nome_run}] AUC={auc:.4f} | Recall={recall:.4f} | F1={f1:.4f} | run_id={run_id[:8]}...')

        return run_id, auc, pipeline

In [ ]:
# rodando três experimentos
experimentos = []

# Experimento 1: Regressão Logística
run_id1, auc1, pipe1 = treinar_e_logar(
    'logistic-regression',
    LogisticRegression(C=1.0, max_iter=1000),
    {'model_type': 'LogisticRegression', 'C': 1.0, 'max_iter': 1000}
)
experimentos.append(('Regressão Logística', auc1, run_id1, pipe1))

# Experimento 2: Random Forest
run_id2, auc2, pipe2 = treinar_e_logar(
    'random-forest',
    RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    {'model_type': 'RandomForest', 'n_estimators': 200, 'max_depth': 10}
)
experimentos.append(('Random Forest', auc2, run_id2, pipe2))

# Experimento 3: Gradient Boosting
run_id3, auc3, pipe3 = treinar_e_logar(
    'gradient-boosting',
    GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42),
    {'model_type': 'GradientBoosting', 'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 4}
)
experimentos.append(('Gradient Boosting', auc3, run_id3, pipe3))

## 3.2 Comparando experimentos <a id="32"></a>

In [ ]:
# consultando os resultados registrados no MLflow
client = mlflow.tracking.MlflowClient()
experimento = client.get_experiment_by_name('telecom-churn-prediction')
runs = client.search_runs(
    experiment_ids=[experimento.experiment_id],
    order_by=['metrics.roc_auc DESC']
)

df_runs = pd.DataFrame([
    {
        'Run':      r.data.tags.get('model', r.info.run_id[:8]),
        'AUC-ROC':  r.data.metrics.get('roc_auc', 0),
        'Recall':   r.data.metrics.get('recall', 0),
        'F1':       r.data.metrics.get('f1_score', 0),
        'Acurácia': r.data.metrics.get('accuracy', 0),
        'Run ID':   r.info.run_id[:8]
    }
    for r in runs
])

print('=== Comparativo de Experimentos — MLflow ===')
df_runs.set_index('Run').round(4)

In [ ]:
# visualizando a comparação
df_plot = df_runs.set_index('Run')[['AUC-ROC', 'Recall', 'F1', 'Acurácia']]

df_plot.plot(
    kind='bar', figsize=(10, 5),
    colormap='Set2', edgecolor='white'
)
plt.title('Comparativo de Experimentos — MLflow Tracking')
plt.ylabel('Score')
plt.xticks(rotation=10, ha='right')
plt.ylim(0.6, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 4. Serialização e Carregamento de Modelos <a id="4"></a>

Para colocar um modelo em produção, precisamos salvá-lo de forma que possa ser carregado por qualquer aplicação — sem depender do ambiente de treinamento.

In [ ]:
# selecionando o melhor modelo (maior AUC-ROC)
melhor = max(experimentos, key=lambda x: x[1])
nome_melhor, auc_melhor, _, pipeline_melhor = melhor

print(f'Melhor modelo: {nome_melhor} (AUC-ROC = {auc_melhor:.4f})')

In [ ]:
# salvando o pipeline completo com joblib
os.makedirs('artifacts', exist_ok=True)
MODEL_PATH = 'artifacts/modelo_churn.joblib'

joblib.dump(pipeline_melhor, MODEL_PATH)
print(f'Modelo salvo em: {MODEL_PATH}')
print(f'Tamanho: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')

In [ ]:
# carregando e verificando que funciona corretamente
modelo_carregado = joblib.load(MODEL_PATH)

y_pred_check = modelo_carregado.predict(X_test)
auc_check    = roc_auc_score(y_test, modelo_carregado.predict_proba(X_test)[:, 1])

print(f'Modelo carregado — AUC-ROC: {auc_check:.4f}')
print('Modelo funcionando corretamente após serialização.')

In [ ]:
# salvando também os metadados do modelo
metadados = {
    'model_name':   nome_melhor,
    'model_path':   MODEL_PATH,
    'features_num': cols_num,
    'features_cat': cols_cat,
    'auc_roc':      round(auc_melhor, 4),
    'dataset':      'TelcoChurn',
    'train_size':   len(X_train),
    'test_size':    len(X_test)
}

with open('artifacts/model_metadata.json', 'w') as f:
    json.dump(metadados, f, indent=2)

print(json.dumps(metadados, indent=2))

# 5. Servindo o Modelo com FastAPI <a id="5"></a>

A forma mais comum de disponibilizar um modelo em produção é como uma **API REST**. Qualquer sistema — site, app mobile, sistema CRM — pode enviar dados e receber previsões em tempo real.

O **FastAPI** é o framework Python mais moderno para isso: rápido, com validação automática de dados e documentação interativa gerada automaticamente.

In [ ]:
# código da API — em produção, isso estaria em um arquivo app.py separado
codigo_api = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(title="Telecom Churn Predictor", version="1.0")

# carregando o modelo na inicialização da API
modelo = joblib.load("artifacts/modelo_churn.joblib")

class ClienteInput(BaseModel):
    tenure: float
    MonthlyCharges: float
    TotalCharges: float
    gender: str
    Partner: str
    Dependents: str
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str

class PrevisaoOutput(BaseModel):
    churn: int
    probabilidade_churn: float
    risco: str

@app.get("/")
def health_check():
    return {"status": "ok", "model": "Telecom Churn Predictor v1.0"}

@app.post("/predict", response_model=PrevisaoOutput)
def predict(cliente: ClienteInput):
    dados = pd.DataFrame([cliente.dict()])
    churn_pred = int(modelo.predict(dados)[0])
    churn_prob = float(modelo.predict_proba(dados)[0][1])

    if churn_prob >= 0.7:
        risco = "alto"
    elif churn_prob >= 0.4:
        risco = "medio"
    else:
        risco = "baixo"

    return PrevisaoOutput(
        churn=churn_pred,
        probabilidade_churn=round(churn_prob, 4),
        risco=risco
    )
'''

# salvando o arquivo da API
with open('artifacts/app.py', 'w') as f:
    f.write(codigo_api)

print('API salva em: artifacts/app.py')

In [ ]:
# iniciando a API em background para testar no Colab
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()

app = FastAPI(title='Telecom Churn Predictor', version='1.0')
modelo_api = joblib.load(MODEL_PATH)

class ClienteInput(BaseModel):
    tenure: float
    MonthlyCharges: float
    TotalCharges: float
    gender: str
    Partner: str
    Dependents: str
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str

@app.get('/')
def health_check():
    return {'status': 'ok', 'model': 'Telecom Churn Predictor v1.0'}

@app.post('/predict')
def predict(cliente: ClienteInput):
    dados = pd.DataFrame([cliente.model_dump()])
    churn_pred = int(modelo_api.predict(dados)[0])
    churn_prob = float(modelo_api.predict_proba(dados)[0][1])
    risco = 'alto' if churn_prob >= 0.7 else ('medio' if churn_prob >= 0.4 else 'baixo')
    return {'churn': churn_pred, 'probabilidade_churn': round(churn_prob, 4), 'risco': risco}

# iniciando em thread separada
thread = threading.Thread(
    target=uvicorn.run,
    kwargs={'app': app, 'host': '0.0.0.0', 'port': 8000, 'log_level': 'error'},
    daemon=True
)
thread.start()

import time
time.sleep(2)
print('API rodando em http://localhost:8000')
print('Documentação interativa: http://localhost:8000/docs')

In [ ]:
import requests

# testando o health check
resp = requests.get('http://localhost:8000/')
print('Health check:', resp.json())

In [ ]:
# testando a previsão com um cliente real do dataset de teste
cliente_exemplo = X_test.iloc[0].to_dict()
# garantindo que TotalCharges não é NaN
if pd.isna(cliente_exemplo.get('TotalCharges')):
    cliente_exemplo['TotalCharges'] = X_test['TotalCharges'].median()

resposta = requests.post('http://localhost:8000/predict', json=cliente_exemplo)
previsao = resposta.json()

print('=== Previsão via API ===')
print(f'Churn previsto:    {previsao["churn"]}')
print(f'Probabilidade:     {previsao["probabilidade_churn"]*100:.1f}%')
print(f'Nível de risco:    {previsao["risco"].upper()}')
print(f'Valor real:        {y_test.iloc[0]}')

In [ ]:
# testando em lote — simulando 10 clientes
print('=== Previsões em lote — 10 clientes ===')
print(f'{"Cliente":<10} {"Risco":<10} {"Probabilidade":<15} {"Real"}')
print('-' * 50)

for i in range(10):
    cliente = X_test.iloc[i].to_dict()
    if pd.isna(cliente.get('TotalCharges')):
        cliente['TotalCharges'] = float(X_test['TotalCharges'].median())
    resp = requests.post('http://localhost:8000/predict', json=cliente).json()
    real = y_test.iloc[i]
    print(f'{i+1:<10} {resp["risco"]:<10} {resp["probabilidade_churn"]*100:.1f}%{"":<9} {real}')

# 6. Monitoramento de Data Drift com Evidently <a id="6"></a>

**Data drift** acontece quando a distribuição dos dados em produção começa a divergir dos dados usados no treinamento. É a causa mais comum de degradação silenciosa de modelos.

**Evidently** é uma biblioteca open-source que gera relatórios automáticos de drift, qualidade dos dados e performance do modelo.

**Cenário:** simularemos que seis meses se passaram e o comportamento dos clientes mudou — talvez por uma campanha agressiva de um concorrente que impactou o perfil dos clientes que chegam à nossa base.

In [ ]:
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import DatasetDriftMetric, ColumnDriftMetric

In [ ]:
# simulando dados de produção com drift
# cenário: concorrente lançou plano mais barato — novos clientes têm tenure menor
# e MonthlyCharges mais baixo (estão em planos básicos)

np.random.seed(99)
n_prod = 500

X_producao = X_test.sample(n_prod, replace=True, random_state=99).copy()

# introduzindo drift: tenure cai 30%, MonthlyCharges cai 20%
X_producao['tenure']         = (X_producao['tenure'] * 0.7 + np.random.normal(0, 2, n_prod)).clip(0).round()
X_producao['MonthlyCharges'] = (X_producao['MonthlyCharges'] * 0.8 + np.random.normal(0, 5, n_prod)).clip(18)

# mais clientes em contratos mensais (month-to-month) — maior churn potencial
mask_contrato = np.random.random(n_prod) < 0.4
X_producao.loc[mask_contrato, 'Contract'] = 'Month-to-month'

print(f'Dados de produção simulados: {X_producao.shape}')

# comparando distribuições
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    ax.hist(X_test[col].dropna(),     bins=30, alpha=0.6, color='steelblue', label='Treino (referência)')
    ax.hist(X_producao[col].dropna(), bins=30, alpha=0.6, color='coral',     label='Produção')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Distribuições: Dados de Referência vs. Produção', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# gerando o relatório de drift com Evidently
relatorio_drift = Report(metrics=[
    DatasetDriftMetric(),
    ColumnDriftMetric(column_name='tenure'),
    ColumnDriftMetric(column_name='MonthlyCharges'),
    ColumnDriftMetric(column_name='TotalCharges'),
])

relatorio_drift.run(
    reference_data=X_test[cols_num].reset_index(drop=True),
    current_data=X_producao[cols_num].reset_index(drop=True)
)

# salvando o relatório HTML
relatorio_drift.save_html('artifacts/drift_report.html')
print('Relatório salvo em: artifacts/drift_report.html')

In [ ]:
# extraindo o resultado do drift em JSON para análise
resultado_drift = relatorio_drift.as_dict()
metricas_drift  = resultado_drift['metrics']

# resumo do drift por coluna
print('=== Relatório de Data Drift ===')
for metrica in metricas_drift:
    nome  = metrica['metric']
    valor = metrica['result']
    if 'drift_detected' in valor:
        col   = valor.get('column_name', 'dataset')
        drift = valor['drift_detected']
        score = valor.get('drift_score', valor.get('share_of_drifted_columns', 'N/A'))
        status = '🚨 DRIFT DETECTADO' if drift else '✅ Estável'
        print(f'  {col:<20} | {status} | Score: {score}')

In [ ]:
# impacto do drift na performance do modelo
y_prod_simulado = (pipeline_melhor.predict_proba(X_producao)[:, 1] > 0.5).astype(int)
taxa_churn_ref  = y_test.mean()
taxa_churn_prod = y_prod_simulado.mean()

print(f'Taxa de churn prevista (referência):  {taxa_churn_ref*100:.1f}%')
print(f'Taxa de churn prevista (produção):    {taxa_churn_prod*100:.1f}%')
print(f'Variação:                             {(taxa_churn_prod - taxa_churn_ref)*100:+.1f}pp')

# visualizando a mudança
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Referência\n(teste)', 'Produção\n(simulada)'],
       [taxa_churn_ref * 100, taxa_churn_prod * 100],
       color=['steelblue', 'coral'], edgecolor='white', width=0.5)
ax.set_ylabel('Taxa de Churn Prevista (%)')
ax.set_title('Impacto do Data Drift na Previsão de Churn')
ax.set_ylim(0, 60)
for i, v in enumerate([taxa_churn_ref * 100, taxa_churn_prod * 100]):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# 7. Pipeline MLOps Completo <a id="7"></a>

Consolidando tudo em um fluxo reprodutível: do dado bruto à previsão monitorada.

In [ ]:
def pipeline_mlops_completo(
    url_dados,
    nome_experimento,
    modelo_clf,
    params_modelo,
    caminho_modelo='artifacts/modelo_producao.joblib'
):
    """
    Pipeline MLOps end-to-end:
    1. Carrega e prepara os dados
    2. Treina com rastreamento MLflow
    3. Serializa o modelo
    4. Verifica drift nos dados de produção
    """
    print('▶ [1/4] Carregando e preparando dados...')
    df = pd.read_csv(url_dados)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    df = df.drop(columns=['customerID'])
    X = df[cols_num + cols_cat]
    y = df['Churn']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f'   Treino: {X_tr.shape} | Teste: {X_te.shape}')

    print('▶ [2/4] Treinando e registrando no MLflow...')
    mlflow.set_experiment(nome_experimento)
    with mlflow.start_run(run_name='pipeline-completo'):
        pipeline = Pipeline([('preprocessor', preprocessor), ('model', modelo_clf)])
        pipeline.fit(X_tr, y_tr)
        y_pred = pipeline.predict(X_te)
        y_prob = pipeline.predict_proba(X_te)[:, 1]
        metricas = {
            'auc_roc':  roc_auc_score(y_te, y_prob),
            'recall':   recall_score(y_te, y_pred),
            'f1_score': f1_score(y_te, y_pred)
        }
        mlflow.log_params(params_modelo)
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(pipeline, 'model')
    print(f'   AUC-ROC: {metricas["auc_roc"]:.4f} | Recall: {metricas["recall"]:.4f}')

    print('▶ [3/4] Serializando o modelo...')
    joblib.dump(pipeline, caminho_modelo)
    print(f'   Modelo salvo em: {caminho_modelo}')

    print('▶ [4/4] Verificando drift (dados de teste como referência)...')
    ref_data  = X_tr[cols_num].reset_index(drop=True)
    curr_data = X_te[cols_num].reset_index(drop=True)
    relatorio = Report(metrics=[DatasetDriftMetric()])
    relatorio.run(reference_data=ref_data, current_data=curr_data)
    resultado = relatorio.as_dict()['metrics'][0]['result']
    drift_detectado = resultado.get('dataset_drift', False)
    print(f'   Drift detectado: {drift_detectado}')

    print('\n✅ Pipeline MLOps concluído com sucesso!')
    return pipeline, metricas


# executando o pipeline completo
pipeline_prod, metricas_prod = pipeline_mlops_completo(
    url_dados='https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv',
    nome_experimento='telecom-churn-pipeline',
    modelo_clf=GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42),
    params_modelo={'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 4}
)

# 8. Conclusão <a id="8"></a>

Nesta aula fechamos o ciclo completo de um projeto de ML em produção:

| Etapa | Ferramenta | O que garante |
|---|---|---|
| Rastreamento de experimentos | MLflow | Reprodutibilidade e comparação de modelos |
| Serialização | joblib | Portabilidade do modelo entre ambientes |
| Metadados | JSON | Documentação do que foi treinado e com quais dados |
| Serving | FastAPI | Previsões em tempo real via API REST |
| Monitoramento | Evidently | Detecção de data drift antes que o modelo degrade |
| Pipeline end-to-end | Python | Reprodutibilidade e automação |

**O ciclo de vida de um modelo em produção:**

```
Dados → Feature Engineering → Treino → Avaliação → Serialização
   ↑                                                      ↓
Retreino ← Drift Detectado ← Monitoramento ← Serving (API)
```

**Boas práticas consolidadas do curso:**

- **Aula 1**: sempre use `Pipeline` para evitar data leakage
- **Aula 2**: avalie com Stratified K-Fold e métricas além da acurácia
- **Aula 3**: decomponha a série antes de modelar
- **Aula 4**: comece com TF-IDF antes de ir para BERT
- **Aula 5**: logar experimentos e monitorar drift não é opcional — é parte do modelo

> Um modelo sem monitoramento é como um piloto automático sem alarme de falha. Funciona — até parar de funcionar.

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)